In [ ]:
#Import necessary libraries, installing wfdb if not already available
import importlib, subprocess, sys
try:
    import wfdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wfdb"])
    importlib.invalidate_caches()
    import wfdb
import matplotlib.pyplot as plt

In [ ]:
# Load ECG data from the MIT-BIH Arrhythmia Database
record = wfdb.rdrecord('100',pn_dir='mitdb')

In [ ]:
# Extract the ECG signal (assuming it's in the first channel)
signal = record.p_signal[:,0]
signal2 = record.p_signal[:,1]

In [ ]:
# Visualize the raw ECG signal
plt.figure(figsize=(12, 4))
plt.plot(signal[:1000])

plt.title('ECG Signal from MIT-BIH Arrhythmia Database')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(signal2[:1000])

plt.title('ECG Signal from MIT-BIH Arrhythmia Database')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.grid()
plt.show()

In [ ]:
# Print signal shape for debugging
print(f"Signal shape: {signal.shape}")

In [ ]:
# Print sampling frequency for debugging
print(record.fs)

In [ ]:
# Calculate and print the duration of the signal
duration_seconds= len(signal) / record.fs
print(f"Duration of the signal: {duration_seconds:.2f} seconds")

In [ ]:
from scipy.signal import butter, filtfilt

In [ ]:
def lowpass_filter(data, cutoff_freq, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = filtfilt(b, a, data)
    return filtered_data

In [ ]:
filtered_signal = lowpass_filter(signal, cutoff_freq=40, fs=record.fs)

In [ ]:
filtered_signal2 = lowpass_filter(signal2, cutoff_freq=40, fs=record.fs)

In [ ]:
plt.figure(figsize=(14, 8))

plt.subplot(2, 1, 1)
plt.plot(signal[:2000])
plt.title('Original ECG Signal')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.grid()

plt.subplot(2, 1, 2)
plt.plot(filtered_signal[:2000])
plt.title('Filtered ECG Signal (Low-pass Filtered)')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(14, 8))
plt.subplot(2, 1, 2)
plt.plot(filtered_signal2[:2000])
plt.title('Filtered ECG Signal (Low-pass Filtered) second channel')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import find_peaks

In [ ]:
import numpy as np

peaks, properties = find_peaks(filtered_signal, distance=int(record.fs * 0.6),height=np.mean(filtered_signal) + 0.5 * np.std(filtered_signal))

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(filtered_signal[:2000], label='Filtered ECG Signal')
plt.plot(peaks[peaks < 2000], filtered_signal[peaks[peaks < 2000]], 'rx', label='Detected R-peaks')
plt.title('ECG Signal with Detected R-peaks')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.legend()
plt.grid()
plt.show()


In [ ]:
num_beats = len(peaks)
print(f"Number of detected heartbeats (R-peaks): {num_beats}")

In [ ]:
# Calculate RR intervals in seconds
rr_intervals = np.diff(peaks) / record.fs
plt.figure(figsize=(10, 4))
plt.plot(rr_intervals, marker='o')
plt.title('RR Intervals from Detected R-peaks')
plt.xlabel('Beat Index')
plt.ylabel('RR Interval (seconds)')
plt.grid()
plt.show()
print(f"Average heart rate: {60 / np.mean(rr_intervals):.2f} bpm")


In [ ]:
sdnn = np.std(rr_intervals)
print(f"SDNN (Standard Deviation of RR Intervals): {sdnn:.4f} seconds")
rmssd = np.sqrt(np.mean(np.diff(rr_intervals)**2))
print(f"RMSSD (Root Mean Square of Successive Differences): {rmssd:.4f} seconds")
plt.figure(figsize=(10, 4))
plt.hist(rr_intervals, bins=15, edgecolor='black')
plt.title('Histogram of RR Intervals')
plt.xlabel('RR Interval (seconds)')
plt.ylabel('Frequency')
plt.grid()
plt.show()

In [ ]:
# Plot RR intervals with mean line
plt.figure(figsize=(12,4))
plt.plot(rr_intervals, marker='o')
plt.axhline(y=np.mean(rr_intervals), color='r', linestyle='--', label=f'Mean RR Interval: {np.mean(rr_intervals):.4f} s')
plt.legend()
plt.title('RR Intervals from Detected R-peaks')
plt.xlabel('Beat Index')
plt.ylabel('RR Interval (seconds)')
plt.grid()
plt.show()
print(f"Mean RR Interval: {np.mean(rr_intervals):.4f} seconds")
print(f"Heart Rate Variability (HRV) Metrics:")
print(f"SDNN= {sdnn:.4f} seconds")
print(f"RMSSD= {rmssd:.4f} seconds")
print(f"Average Heart Rate= {60 / np.mean(rr_intervals):.2f} bpm")


In [ ]:
outliers = rr_intervals[(rr_intervals < np.mean(rr_intervals) - 2 * sdnn) | (rr_intervals > np.mean(rr_intervals) + 2 * sdnn)]
print(f"Outlier RR Intervals: {outliers}")
print(f"Number of Outliers: {len(outliers)}")

In [ ]:
from scipy.stats import skew, kurtosis
from scipy.stats import skew, kurtosis
from scipy.fft import fft, fftfreq

In [ ]:
ecg_fft = fft(filtered_signal[:5000])
w = fftfreq(len(filtered_signal[:5000]), d=1/record.fs)
plt.figure(figsize=(12, 6))
plt.plot(w[:250], np.abs(ecg_fft)[:250])
plt.title('Frequency Spectrum of Filtered ECG Signal')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.grid()
plt.show()
print(f"Skewness of RR Intervals: {skew(rr_intervals):.4f}")
print(f"Kurtosis of RR Intervals: {kurtosis(rr_intervals):.4f}")

In [ ]:
from scipy.signal import welch
frequencies, power_spectrum = welch(rr_intervals, fs=1/np.mean(rr_intervals), nperseg=len(rr_intervals))
plt.figure(figsize=(12, 6))
plt.semilogy(frequencies, power_spectrum)
plt.title('Power Spectral Density of RR Intervals')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power')
plt.grid()
plt.show()

In [ ]:
if_band = (frequencies >= 0.04) & (frequencies < 0.15)
hf_band = (frequencies >= 0.15) & (frequencies < 0.4)


In [ ]:
lf_power = np.trapz(power_spectrum[if_band], frequencies[if_band])
hf_power = np.trapz(power_spectrum[hf_band], frequencies[hf_band])

In [ ]:
print(f"LF Power: {lf_power:.6f}")
print(f"HF Power: {hf_power:.6f}")
print(f"LF/HF Ratio: {lf_power/hf_power:.6f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
import pandas as pd
# Create a DataFrame for HRV features
data = {
    "BPM": [72, 75, 90, 110, 55, 130, 68, 95],
    "SDNN": [0.08, 0.07, 0.05, 0.02, 0.09, 0.01, 0.07, 0.03],
    "LF_HF": [1.2, 1.0, 2.5, 4.0, 0.8, 5.0, 1.1, 3.5],
    "Label": [0, 0, 0, 1, 0, 1, 0, 1]
}
df = pd.DataFrame(data)
print(df)
X = df[["BPM", "SDNN", "LF_HF"]]
y = df["Label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
y_predections = model.predict(X_test)
print(f"Predicted labels: {y_predections}")
accuracy = accuracy_score(y_test, y_predections)   
print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:")
print(classification_report(y_test, y_predections))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_predections))


In [ ]:
def predict_arrhythmia(bpm, sdnn, lf_hf):
    features = [[bpm, sdnn, lf_hf]]
    prediction = model.predict(features)
    return "Arrhythmia Detected" if prediction[0] == 1 else "Normal"
# Example usage
result = predict_arrhythmia(85, 0.06, 2.0)
print(result)


In [ ]:
def extract_features(
    filtered_signal,
    peaks,
    rr_intervals,
    power_spectrum,
    frequencies
):

    features = {}

    # Heart Rate
    features['BPM'] = 60 / np.mean(rr_intervals)

    # HRV Metrics
    features['SDNN'] = np.std(rr_intervals)

    features['RMSSD'] = np.sqrt(
        np.mean(np.diff(rr_intervals)**2)
    )

    # Frequency Bands
    lf_band = (
        (frequencies >= 0.04)
        & (frequencies < 0.15)
    )

    hf_band = (
        (frequencies >= 0.15)
        & (frequencies < 0.4)
    )

    # Spectral Power
    lf_power = np.trapz(
        power_spectrum[lf_band],
        frequencies[lf_band]
    )

    hf_power = np.trapz(
        power_spectrum[hf_band],
        frequencies[hf_band]
    )

    # LF/HF Ratio
    features['LF_HF'] = lf_power / hf_power

    # Statistical Features
    features['Skewness'] = skew(rr_intervals)

    features['Kurtosis'] = kurtosis(rr_intervals)

    return features
features = extract_features(
    filtered_signal,
    peaks,
    rr_intervals,
    power_spectrum,
    frequencies
)
print(features)

In [ ]:
feature_df = pd.DataFrame([features])
print(feature_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import welch
from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# =========================
# FEATURE EXTRACTION
# =========================

def extract_features(
    rr_intervals,
    power_spectrum,
    frequencies
):

    features = {}

    # Heart Rate
    features["BPM"] = (
        60 / np.mean(rr_intervals)
    )

    # HRV Metrics
    features["SDNN"] = np.std(rr_intervals)

    features["RMSSD"] = np.sqrt(
        np.mean(np.diff(rr_intervals) ** 2)
    )

    # Frequency Bands
    lf_band = (
        (frequencies >= 0.04)
        & (frequencies < 0.15)
    )

    hf_band = (
        (frequencies >= 0.15)
        & (frequencies < 0.4)
    )

    # Spectral Power
    lf_power = np.trapz(
        power_spectrum[lf_band],
        frequencies[lf_band]
    )

    hf_power = np.trapz(
        power_spectrum[hf_band],
        frequencies[hf_band]
    )

    # LF/HF Ratio
    features["LF_HF"] = (
        lf_power / hf_power
    )

    # Statistical Features
    features["Skewness"] = skew(
        rr_intervals
    )

    features["Kurtosis"] = kurtosis(
        rr_intervals
    )

    return features


# =========================
# CREATE DATASET
# =========================

dataset = []

for i in range(50):

    simulated_features = {
        "BPM": np.random.normal(75, 15),

        "SDNN": max(
            0.01,
            np.random.normal(0.07, 0.03)
        ),

        "RMSSD": max(
            0.01,
            np.random.normal(0.04, 0.02)
        ),

        "LF_HF": max(
            0.1,
            np.random.normal(2.0, 1.5)
        ),

        "Skewness": np.random.normal(
            0,
            1
        ),

        "Kurtosis": np.random.normal(
            3,
            1
        )
    }

    # Simple educational labeling
    if (
        simulated_features["BPM"] > 100
        or simulated_features["LF_HF"] > 3
    ):

        simulated_features["Label"] = 1

    else:

        simulated_features["Label"] = 0

    dataset.append(simulated_features)

# Convert to DataFrame
df = pd.DataFrame(dataset)

print(df.head())


# =========================
# MACHINE LEARNING
# =========================

X = df[
    [
        "BPM",
        "SDNN",
        "RMSSD",
        "LF_HF",
        "Skewness",
        "Kurtosis"
    ]
]

y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

# Create Model
model = LogisticRegression()

# Train Model
model.fit(
    X_train,
    y_train
)

# Predictions
y_predictions = model.predict(
    X_test
)

# Evaluation
accuracy = accuracy_score(
    y_test,
    y_predictions
)

print(f"\nAccuracy: {accuracy:.2f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_predictions
    )
)


# =========================
# ARRHYTHMIA PREDICTION
# =========================

def predict_arrhythmia(
    bpm,
    sdnn,
    rmssd,
    lf_hf,
    skewness,
    kurtosis
):

    input_features = [[
        bpm,
        sdnn,
        rmssd,
        lf_hf,
        skewness,
        kurtosis
    ]]

    prediction = model.predict(
        input_features
    )

    probability = model.predict_proba(
        input_features
    )

    if prediction[0] == 1:

        result = "Possible Arrhythmia Detected"

    else:

        result = "Normal Rhythm"

    return result, probability


# =========================
# TEST PREDICTION
# =========================

result, probability = predict_arrhythmia(
    bpm=features["BPM"],
    sdnn=features["SDNN"],
    rmssd=features["RMSSD"],
    lf_hf=features["LF_HF"],
    skewness=features["Skewness"],
    kurtosis=features["Kurtosis"]
)

print("\nPrediction Result:")
print(result)

print("\nPrediction Probabilities:")
print(probability)

In [ ]:
def interpret_features(features):

    interpretations = []

    if features["BPM"] > 100:
        interpretations.append(
            "Elevated BPM may indicate tachycardia."
        )

    if features["SDNN"] < 0.05:
        interpretations.append(
            "Low SDNN suggests reduced heart rate variability."
        )

    if features["RMSSD"] < 0.03:
        interpretations.append(
            "Low RMSSD may reflect reduced parasympathetic activity."
        )

    if features["LF_HF"] > 3:
        interpretations.append(
            "High LF/HF ratio may indicate sympathetic dominance."
        )

    return interpretations

In [ ]:
prediction, probability = predict_arrhythmia(
    bpm=features["BPM"],
    sdnn=features["SDNN"],
    rmssd=features["RMSSD"],
    lf_hf=features["LF_HF"],
    skewness=features["Skewness"],
    kurtosis=features["Kurtosis"]
)

interpretation = interpret_features(features)